# MNIST 매니폴드 학습: t-SNE · UMAP · Isomap · LLE (3D Interactive)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gshong-ai/ADA26/blob/claude/mnist-dimensionality-reduction-H2Meg/03_manifold_learning/mnist_manifold_3d.ipynb)

## 개요

PCA·LDA·MDS가 **선형 구조**를 보존하는 데 집중한다면,  
이번 실습의 기법들은 데이터가 고차원에 숨겨진 **비선형 매니폴드 구조**를 펼쳐냅니다.

| 기법 | 유형 | 핵심 아이디어 | 특징 |
|------|------|--------------|------|
| **t-SNE** | 비지도 / 비선형 | 고차원 유사도 → 저차원 확률 분포 매칭 | 클러스터 내부 구조에 강함 |
| **UMAP** | 비지도 / 비선형 | 리만 기하 기반 위상 구조 보존 | t-SNE보다 빠르고 전역 구조도 보존 |
| **Isomap** | 비지도 / 비선형 | 측지 거리(그래프 최단 경로) 기반 MDS | 매니폴드 위 실제 경로 거리 보존 |
| **LLE** | 비지도 / 비선형 | 이웃 재구성 가중치 보존 | 국소 선형 구조 보존 |

> **이전 노트북**: `02_pca_lda_mds_comparison/` — PCA · LDA · MDS 비교

## 0. 패키지 설치

In [ ]:
# Colab / 로컬 환경 모두 대응
import sys
!pip install -q plotly scikit-learn umap-learn

# tensorflow: MNIST 데이터 로드용 (keras.datasets)
!pip install -q tensorflow

## 1. 라이브러리 임포트

In [ ]:
# ─── 라이브러리 임포트 ───
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap, LocallyLinearEmbedding
from sklearn.preprocessing import StandardScaler

import umap

from tensorflow import keras

# Colab 인라인 렌더링
pio.renderers.default = 'colab'

print('라이브러리 로드 완료')

## 2. 하이퍼파라미터 & 설정

In [ ]:
# ─── 하이퍼파라미터 & 전역 설정 ───
RANDOM_SEED    = 42
N_SAMPLES      = 3000   # 전체 사용 샘플 수 (클래스당 300개)
N_PCA_PRETRAIN = 50     # 사전 PCA 압축 차원 (784 → 50, 속도 개선용)
OUTPUT_HTML    = 'mnist_manifold_3d.html'

# ─── t-SNE 파라미터 ───
TSNE_PERPLEXITY   = 30
TSNE_MAX_ITER     = 1000
TSNE_LEARNING_RATE = 200

# ─── UMAP 파라미터 ───
UMAP_N_NEIGHBORS  = 15
UMAP_MIN_DIST     = 0.1

# ─── Isomap 파라미터 ───
ISOMAP_N_NEIGHBORS = 10

# ─── LLE 파라미터 ───
LLE_N_NEIGHBORS = 10
LLE_METHOD      = 'standard'   # 'standard' | 'modified' | 'hessian' | 'ltsa'

# ─── 시각화 설정 ───
COLORS = [
    '#E74C3C',  # 0 — 빨강
    '#3498DB',  # 1 — 파랑
    '#2ECC71',  # 2 — 초록
    '#F39C12',  # 3 — 주황
    '#9B59B6',  # 4 — 보라
    '#1ABC9C',  # 5 — 청록
    '#E67E22',  # 6 — 진주황
    '#34495E',  # 7 — 짙은 회색
    '#E91E63',  # 8 — 핑크
    '#00BCD4',  # 9 — 하늘색
]

print('설정 완료')
print(f'  샘플 수: {N_SAMPLES} (클래스당 {N_SAMPLES // 10}개)')
print(f'  사전 PCA 차원: 784 → {N_PCA_PRETRAIN}')

## 3. 데이터 로드 & 전처리

클래스 불균형을 방지하기 위해 **클래스별 균등 서브샘플링**을 적용합니다.  
그 후 **사전 PCA**(784 → 50차원)로 압축해 비선형 기법의 계산 속도를 높입니다.

In [ ]:
# ─── MNIST 로드 ───
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_all = np.concatenate([x_train, x_test], axis=0).reshape(-1, 784).astype('float32') / 255.0
y_all = np.concatenate([y_train, y_test], axis=0)

print(f'전체 MNIST: {len(x_all):,}개  |  shape: {x_all.shape}')

# ─── 클래스별 균등 서브샘플링 ───
rng = np.random.default_rng(RANDOM_SEED)
per_class = N_SAMPLES // 10
idx_list = []
for digit in range(10):
    ci = np.where(y_all == digit)[0]
    idx_list.append(rng.choice(ci, size=per_class, replace=False))

idx = np.concatenate(idx_list)
rng.shuffle(idx)

x, y = x_all[idx], y_all[idx]
print(f'서브샘플: {len(x):,}개  (클래스당 {per_class}개)')

# ─── 사전 PCA 압축 (784 → 50차원) ───
# 비선형 기법의 O(n²) ~ O(n³) 계산량을 줄이기 위한 전처리
# PCA로 먼저 정보 손실 최소화 압축 후 매니폴드 학습 적용
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

pca_pre = PCA(n_components=N_PCA_PRETRAIN, random_state=RANDOM_SEED)
x_pca50 = pca_pre.fit_transform(x_scaled)

retained_var = pca_pre.explained_variance_ratio_.sum()
print(f'사전 PCA: 784 → {N_PCA_PRETRAIN}차원  (분산 보존율: {retained_var:.3f} = {retained_var*100:.1f}%)')

## 4. t-SNE — t-분포 확률적 이웃 임베딩

**핵심 아이디어:**  
고차원 공간에서 가까운 점들의 **유사도를 가우시안 확률**로, 저차원에서는 **t-분포 확률**로 나타낸 뒤  
두 분포의 KL 발산을 최소화하는 방향으로 점들을 배치합니다.

**t-분포를 쓰는 이유:**  
가우시안보다 꼬리가 두꺼워서 먼 점들을 더 멀리 밀어내는 "군집화 힘"이 강합니다.  
→ 클러스터 간 분리가 선명해지는 "군집화 편향(crowding problem 해결)".

| 파라미터 | 의미 | 권장값 |
|----------|------|--------|
| `perplexity` | 효과적 이웃 수 (유사도 폭 조절) | 5 ~ 50 |
| `learning_rate` | 최적화 보폭 | 10 ~ 1000 |
| `max_iter` | 반복 횟수 | 500 ~ 2000 |

> ⚠️ t-SNE는 **전역 구조를 보존하지 않습니다.** 클러스터 간 거리는 의미 없음.

In [ ]:
# ─── t-SNE 3D 투영 ───
print('[t-SNE] 3D 투영 중... (사전 PCA 50차원 입력)')
print(f'  perplexity={TSNE_PERPLEXITY}, max_iter={TSNE_MAX_ITER}, lr={TSNE_LEARNING_RATE}')

t0 = time.time()
tsne = TSNE(
    n_components=3,
    perplexity=TSNE_PERPLEXITY,
    max_iter=TSNE_MAX_ITER,
    learning_rate=TSNE_LEARNING_RATE,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
tsne_coords = tsne.fit_transform(x_pca50)
tsne_elapsed = time.time() - t0

print(f'  완료 ({tsne_elapsed:.1f}s)  |  KL 발산: {tsne.kl_divergence_:.4f}')

## 5. UMAP — 균일 매니폴드 근사 및 투영

**핵심 아이디어:**  
데이터가 **리만 다양체(Riemannian manifold)** 위에 균일하게 분포한다고 가정하고,  
고차원의 **위상 구조(fuzzy simplicial set)**를 저차원에서 최대한 보존합니다.

**t-SNE 대비 장점:**
- 훨씬 빠름 (대규모 데이터에 적합)
- **전역 구조도 어느 정도 보존** (클러스터 간 거리에 의미)
- 새 데이터를 변환(transform)할 수 있음

| 파라미터 | 의미 | 권장값 |
|----------|------|--------|
| `n_neighbors` | 지역 구조 반경 (클수록 전역 구조 강조) | 5 ~ 50 |
| `min_dist` | 저차원 클러스터 밀집도 (작을수록 촘촘) | 0.0 ~ 0.5 |

In [ ]:
# ─── UMAP 3D 투영 ───
print('[UMAP] 3D 투영 중... (사전 PCA 50차원 입력)')
print(f'  n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST}')

t0 = time.time()
umap_model = umap.UMAP(
    n_components=3,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    random_state=RANDOM_SEED,
)
umap_coords = umap_model.fit_transform(x_pca50)
umap_elapsed = time.time() - t0

print(f'  완료 ({umap_elapsed:.1f}s)')

## 6. Isomap — 등거리 매핑

**핵심 아이디어:**  
유클리드 거리 대신 **매니폴드 위의 측지 거리(geodesic distance)**를 사용하는 MDS입니다.

1. k-최근접 이웃 그래프 구성
2. **다익스트라 알고리즘**으로 모든 쌍의 그래프 최단 경로(측지 거리) 계산
3. 이 거리 행렬에 MDS 적용

**직관:**  
두루마리(Swiss Roll)에서 두 점의 유클리드 거리는 롤을 관통하지만,  
측지 거리는 롤의 표면을 따라가는 실제 거리입니다.

| 파라미터 | 의미 | 권장값 |
|----------|------|--------|
| `n_neighbors` | 이웃 그래프 연결 수 | 5 ~ 20 |

> ⚠️ 그래프가 끊어지면 실패합니다. n_neighbors가 너무 작으면 연결 오류 발생.

In [ ]:
# ─── Isomap 3D 투영 ───
print('[Isomap] 3D 투영 중... (사전 PCA 50차원 입력)')
print(f'  n_neighbors={ISOMAP_N_NEIGHBORS}')

t0 = time.time()
isomap = Isomap(
    n_components=3,
    n_neighbors=ISOMAP_N_NEIGHBORS,
    n_jobs=-1,
)
isomap_coords = isomap.fit_transform(x_pca50)
isomap_elapsed = time.time() - t0

print(f'  완료 ({isomap_elapsed:.1f}s)  |  재구성 오차: {isomap.reconstruction_error():.4f}')

## 7. LLE — 국소 선형 임베딩

**핵심 아이디어:**  
각 점은 **이웃 점들의 선형 결합**으로 재구성될 수 있다고 가정합니다.

1. 각 점에서 k-이웃의 **재구성 가중치** W 계산 (고차원)
2. 저차원에서 동일한 가중치 W가 성립하도록 좌표 최적화

**직관:**  
"옆에 있는 점들과의 상대적 위치 관계"를 그대로 저차원에 복사합니다.

| 파라미터 | 의미 | 권장값 |
|----------|------|--------|
| `n_neighbors` | 재구성에 쓸 이웃 수 | 5 ~ 20 |
| `method` | 변형 알고리즘 | `'standard'` / `'modified'` / `'hessian'` / `'ltsa'` |

> ⚠️ LLE는 **국소 구조**에 매우 민감합니다. 노이즈나 이웃 수 설정에 따라 결과가 크게 달라집니다.

In [ ]:
# ─── LLE 3D 투영 ───
print('[LLE] 3D 투영 중... (사전 PCA 50차원 입력)')
print(f'  n_neighbors={LLE_N_NEIGHBORS}, method={LLE_METHOD!r}')

t0 = time.time()
lle = LocallyLinearEmbedding(
    n_components=3,
    n_neighbors=LLE_N_NEIGHBORS,
    method=LLE_METHOD,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
lle_coords = lle.fit_transform(x_pca50)
lle_elapsed = time.time() - t0

print(f'  완료 ({lle_elapsed:.1f}s)  |  재구성 오차: {lle.reconstruction_error_:.6f}')

## 8. 인터랙티브 3D 비교 시각화

4개 기법의 결과를 한 화면에서 비교합니다.  
**마우스 드래그**: 회전 · **스크롤**: 줌 · **호버**: 값 확인 · **범례 클릭**: 클래스 토글

In [ ]:
# ─── 3D 산점도 트레이스 추가 헬퍼 ───
def add_scatter3d(fig, coords, labels, row, col, show_legend):
    """숫자 클래스별 3D 산점도를 subplot에 추가"""
    for digit in range(10):
        mask = labels == digit
        fig.add_trace(
            go.Scatter3d(
                x=coords[mask, 0],
                y=coords[mask, 1],
                z=coords[mask, 2],
                mode='markers',
                name=f'숫자 {digit}',
                legendgroup=f'digit_{digit}',
                showlegend=show_legend,
                marker=dict(
                    size=2.5,
                    color=COLORS[digit],
                    opacity=0.75,
                    line=dict(width=0),
                ),
                hovertemplate=(
                    f'<b>숫자 {digit}</b><br>'
                    'x: %{x:.3f}<br>'
                    'y: %{y:.3f}<br>'
                    'z: %{z:.3f}'
                    '<extra></extra>'
                ),
            ),
            row=row, col=col,
        )


# ─── 2×2 subplot Figure 구성 ───
fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{'type': 'scatter3d'}, {'type': 'scatter3d'}],
        [{'type': 'scatter3d'}, {'type': 'scatter3d'}],
    ],
    subplot_titles=[
        f't-SNE  (KL발산={tsne.kl_divergence_:.3f},  {tsne_elapsed:.1f}s)',
        f'UMAP  ({umap_elapsed:.1f}s)',
        f'Isomap  (재구성오차={isomap.reconstruction_error():.3f},  {isomap_elapsed:.1f}s)',
        f'LLE  (재구성오차={lle.reconstruction_error_:.4f},  {lle_elapsed:.1f}s)',
    ],
    vertical_spacing=0.05,
    horizontal_spacing=0.02,
)

add_scatter3d(fig, tsne_coords,   y, row=1, col=1, show_legend=True)
add_scatter3d(fig, umap_coords,   y, row=1, col=2, show_legend=False)
add_scatter3d(fig, isomap_coords, y, row=2, col=1, show_legend=False)
add_scatter3d(fig, lle_coords,    y, row=2, col=2, show_legend=False)

# ─── 공통 축 스타일 ───
axis_style = dict(
    backgroundcolor='#F0F2F5',
    gridcolor='white',
    showticklabels=False,
)
scene_defaults = dict(
    xaxis=dict(title='', **axis_style),
    yaxis=dict(title='', **axis_style),
    zaxis=dict(title='', **axis_style),
    bgcolor='#F0F2F5',
    camera=dict(eye=dict(x=1.6, y=1.6, z=1.0)),
)

fig.update_layout(
    title=dict(
        text=(
            'MNIST 매니폴드 학습 비교: t-SNE · UMAP · Isomap · LLE  (3D Interactive)<br>'
            '<sup>마우스 드래그: 회전 · 스크롤: 줌 · 호버: 값 확인 · 범례 클릭: 클래스 토글</sup>'
        ),
        x=0.5,
        font=dict(size=17),
    ),
    scene=scene_defaults,
    scene2=scene_defaults,
    scene3=scene_defaults,
    scene4=scene_defaults,
    legend=dict(
        title=dict(text='숫자 클래스', font=dict(size=13)),
        itemsizing='constant',
        font=dict(size=12),
        x=1.01, y=0.5,
    ),
    width=1400,
    height=1100,
    margin=dict(l=0, r=130, t=100, b=0),
    paper_bgcolor='white',
)

fig.show()

fig.write_html(OUTPUT_HTML, include_plotlyjs='cdn')
print(f'✅ {OUTPUT_HTML} 저장 완료')

## 9. 성능 비교 요약

In [ ]:
# ─── 실루엣 스코어 계산 및 종합 비교 ───
from sklearn.metrics import silhouette_score

print('실루엣 스코어 계산 중... (클래스 분리도 지표, 1에 가까울수록 좋음)')

sil_tsne   = silhouette_score(tsne_coords,   y, sample_size=1000, random_state=RANDOM_SEED)
sil_umap   = silhouette_score(umap_coords,   y, sample_size=1000, random_state=RANDOM_SEED)
sil_isomap = silhouette_score(isomap_coords, y, sample_size=1000, random_state=RANDOM_SEED)
sil_lle    = silhouette_score(lle_coords,    y, sample_size=1000, random_state=RANDOM_SEED)

methods = ['t-SNE', 'UMAP', 'Isomap', 'LLE']
sil_scores = [sil_tsne, sil_umap, sil_isomap, sil_lle]
elapsed = [tsne_elapsed, umap_elapsed, isomap_elapsed, lle_elapsed]

print()
print('=' * 55)
print(f'{"기법":>8}  {"실루엣 스코어":>14}  {"소요 시간":>10}')
print('-' * 55)
for name, sil, t in zip(methods, sil_scores, elapsed):
    best_mark = ' ★' if sil == max(sil_scores) else ''
    fast_mark = ' ⚡' if t == min(elapsed) else ''
    print(f'{name:>8}  {sil:>14.4f}  {t:>9.1f}s{best_mark}{fast_mark}')
print('=' * 55)
print('★ = 최고 실루엣 스코어 (클래스 분리 우수)')
print('⚡ = 최고 속도')

In [ ]:
# ─── 비교 막대 차트 ───
import plotly.express as px

method_colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

fig_bar = go.Figure()

# 실루엣 스코어 비교
fig_bar.add_trace(go.Bar(
    name='실루엣 스코어',
    x=methods,
    y=sil_scores,
    marker_color=method_colors,
    opacity=0.85,
    text=[f'{s:.4f}' for s in sil_scores],
    textposition='auto',
))

fig_bar.update_layout(
    title='매니폴드 학습 기법 비교 — 실루엣 스코어 (MNIST 10-class)',
    xaxis_title='기법',
    yaxis_title='실루엣 스코어 (↑ 클수록 좋음)',
    yaxis=dict(range=[0, 1]),
    width=700, height=450,
    showlegend=False,
)
fig_bar.show()

# 속도 비교
fig_time = go.Figure(go.Bar(
    x=methods,
    y=elapsed,
    marker_color=method_colors,
    opacity=0.85,
    text=[f'{t:.1f}s' for t in elapsed],
    textposition='auto',
))
fig_time.update_layout(
    title='매니폴드 학습 기법 비교 — 소요 시간 (↓ 작을수록 빠름)',
    xaxis_title='기법',
    yaxis_title='소요 시간 (초)',
    width=700, height=450,
    showlegend=False,
)
fig_time.show()

## 10. 결론 및 기법 선택 가이드

| 기법 | 강점 | 약점 | 언제 쓰나 |
|------|------|------|-----------|
| **t-SNE** | 클러스터 분리 선명 | 느림, 전역 구조 비보존, 재현성 낮음 | 클러스터 탐색, 발표용 시각화 |
| **UMAP** | 빠름, 전역+국소 구조 보존, transform 가능 | 파라미터 민감, 확률적 | 탐색적 분석, 대규모 데이터 |
| **Isomap** | 측지 거리 기반 매니폴드 복원 | 그래프 단절 취약, 느림 | 연속적인 매니폴드 구조 복원 |
| **LLE** | 국소 선형 구조 정밀 보존 | 노이즈 취약, 전역 구조 비보존 | 국소 관계 중심 분석 |

### 핵심 교훈

> **비선형 매니폴드 학습은 "어디를 바라보느냐"의 차이입니다.**  
> - t-SNE · UMAP → 확률 분포 / 위상 관점  
> - Isomap → 측지 거리 관점  
> - LLE → 이웃 재구성 관점  
> 
> 실무에서는 **UMAP을 기본값**으로 쓰고, 클러스터 구조가 중요하면 t-SNE를 추가 확인하는 전략을 권장합니다.